# Lesson 4: Drive AGA from the neo4j driver

**Module:** AGA Fundamentals  
**Dataset:** Movies (Actors, Movies, Genres, Users)

This notebook uses the plain `neo4j` driver — no `graphdatascience` package.


## Open the driver

Open a single `neo4j` driver against your AuraDB instance. It stays open for the whole workflow.


In [ ]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()

uri      = os.getenv('AURA_URI')
username = os.getenv('AURA_USERNAME')
password = os.getenv('AURA_PASSWORD')

driver = GraphDatabase.driver(
    uri,
    auth=(username, password),
)
driver.verify_connectivity()
print("Driver ready.")


## Create an AGA session

`gds.session.getOrCreate` is Cypher. Send it through the driver like any other query and capture the returned `sessionId`.


In [ ]:
with driver.session() as session:
    record = session.run(
        """
        CALL gds.session.getOrCreate(
          'driver-demo',
          '2GB',
          duration({minutes: 30})
        )
        YIELD id AS sessionId, name, status
        RETURN sessionId, name, status
        """
    ).single()
    aga_session_id = record["sessionId"]

print(f"Session ready: {aga_session_id}")


## Project the actor-collaboration graph

The Cypher projection is the same one from Module 2 Lesson 1. Pass the AGA session id in as a query parameter.


In [ ]:
with driver.session() as session:
    result = session.run(
        """
        CYPHER runtime=parallel
        MATCH (source:Actor)-[r:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(target:Actor)
        WHERE source <> target
        WITH source, target, count(r) AS collaborations
        WITH gds.graph.project(
          'actor-collaborations',
          source, target,
          {
            sourceNodeLabels: labels(source),
            targetNodeLabels: labels(target),
            relationshipType: 'COLLABORATED',
            relationshipProperties: {collaborations: collaborations}
          },
          { sessionId: $sessionId }
        ) AS g
        RETURN g.graphName AS name, g.nodeCount AS nodes, g.relationshipCount AS rels
        """,
        sessionId=aga_session_id,
    ).single()

print(f"Projected {result['name']}: {result['nodes']:,} nodes, {result['rels']:,} rels")


## Run degree centrality

Algorithm calls are ordinary Cypher too. Stream the top 10 most-connected actors.


In [ ]:
with driver.session() as session:
    records = session.run(
        """
        CALL gds.degree.stream('actor-collaborations', {relationshipWeightProperty: 'collaborations'})
        YIELD nodeId, score
        WITH gds.util.asNode(nodeId) AS actor, score
        RETURN actor.name AS name, score
        ORDER BY score DESC
        LIMIT 10
        """
    )
    for row in records:
        print(f"{row['score']:>6.2f}  {row['name']}")


## Write results back to AuraDB

`write` persists the score back to each Actor node.


In [ ]:
with driver.session() as session:
    record = session.run(
        """
        CALL gds.degree.write('actor-collaborations', {
          writeProperty: 'degree',
          relationshipWeightProperty: 'collaborations'
        })
        YIELD nodePropertiesWritten
        RETURN nodePropertiesWritten
        """
    ).single()

print(f"Wrote degree to {record['nodePropertiesWritten']:,} actors.")


## Clean up

Delete the AGA session and close the driver.


In [ ]:
with driver.session() as session:
    session.run("CALL gds.session.delete('driver-demo')")

print("Session deleted.")

driver.close()


That's it — you've driven AGA end to end from the plain `neo4j` driver.

The exact same Cypher runs from any Neo4j driver — Java, .NET, Go, JavaScript. Reach for this pattern when you're embedding AGA into an existing service, or when the `graphdatascience` client isn't the right dependency for your codebase.
